# Flexible Job Shop Scheduling Problem with Transportation (FJSPT)

**Цель:** Минимизировать общую продолжительность выполнения всех работ (makespan) — .

### 1. Основные компоненты

* **Работы ($J$):** Множество из  работ. Каждая работа состоит из последовательности (маршрута) операций .
* **Оборудование ($M$):** Множество из  станков.
* **Транспорт ($V$):** Множество из  транспортных средств (ТС).

### 2. Условия выполнения операций

* **Производственные операции ($u$):** Каждая операция может быть выполнена на любом станке из заданного подмножества $M_u \subseteq M$. Время обработки $p^a_u$ зависит от выбранного станка.
* **Транспортные операции ($t_u$):** Каждой производственной операции $u$ предшествует транспортировка $t_u$ (включая первую операцию — из депо LU к станку). Транспортировка после последней операции не учитывается.
* **Особенности ТС:** Любое ТС может выполнить любую транспортную операцию ($V^{t_u} = V$). Время перевозки $p^J_{t_u}$ зависит только от расстояния и не зависит от выбора ТС.

### 3. Логистика и временные затраты

* **Перемещения:** Учитывается время перемещения как с грузом, так и без него (порожний пробег). Время в пути между станками $m_a$ и $m_b$ обозначается как $d(m_a, m_b)$.
* **Правило треугольника:** Транспортные времена удовлетворяют неравенству треугольника: $d(m_1, m_2) \leqslant d(m_1, m_3) + d(m_3, m_2)$.
* **Нулевая транспортировка:** Если две последовательные операции одной работы выполняются на одном и том же станке, транспортная операция между ними отсутствует.
* **Общее время ТС:** Время занятости ТС ($p^M_{t_u}$) складывается из времени перевозки груза и времени порожнего пробега до следующего места погрузки.

### Основные ограничения FJSPT

* **Технологическая последовательность:** Операции одной работы $j$ должны выполняться строго в заданном порядке. Операция $u + 1$ не может начаться, пока не завершится операция $u$ и последующая транспортировка $t_{u+1}$.
* **Неразрывность операций:** После начала операции на станке она не может быть прервана (non-preemption).
* **Ограничение ресурсов (станки):** Каждый станок в любой момент времени может обрабатывать не более одной операции.
* **Ограничение ресурсов (транспорт):** Каждое транспортное средство (ТС) может перевозить только одну работу за раз.
* **Допустимость оборудования:** Операция  может быть назначена только на станок из подмножества $M^u$.
* **Транспортная логистика:** ТС должно сначала завершить текущую доставку, затем доехать до местонахождения следующей работы (порожний пробег), и только после этого начать новую транспортировку.

## Обучение модели для FJSPT с использованием подхода RL4CO

In [5]:
from pytorch_lightning.loggers import CometLogger
import torch
import time
from IPython.display import clear_output

from lightning.pytorch.callbacks import ModelCheckpoint, RichModelSummary

from env import FJSPTEnv
from rl4co.utils.trainer import RL4COTrainer

from model import L2DModel
from l2d_policy import L2DPolicy

import matplotlib.pyplot as plt

In [6]:
# Окружение для тестирования модели с помощью подготовленных датасетов
# env = FJSPTEnv(
#     generator_params={
#       # датасеты с данными FJSP
#       # нужно сделать возможность указывать пути инстансов c одинаковыми операциями, доступных машинами и временами обработки
#       # смотреть datasets/schemas.txt !!!
#       # либо вручную разложить инстансы по папкам c одинаковыми операциями, доступных машинами и временами обработки
#       "proc_file_path": "../datasets/1_Deroussi_and_Norre/proc_data",  
#       # датасет с временем транспортировки 
#       "trucks_file_path": "../datasets/1_Deroussi_and_Norre/trucks_data/trucks_layout8.txt",
#     },
# )

In [7]:
from rl4co.envs import ENV_REGISTRY

ENV_REGISTRY["fjspt"] = FJSPTEnv

env = FJSPTEnv(
    generator_params={
      "num_jobs": 20,  # the total number of jobs
      "num_machines": 10,  # the total number of machines that can process operations
      "num_trucks": 2,  # the total number of trucks
      "min_ops_per_job": 15,  # minimum number of operatios per job
      "max_ops_per_job": 25,  # maximum number of operations per job
      "min_processing_time": 20,  # the minimum time required for a machine to process an operation
      "max_processing_time": 70,  # the maximum time required for a machine to process an operation
      "min_transportation_time": 0,  # the minimum time required for a truck to transport
      "max_transportation_time": 50,  # the maximum time required for a truck to transport
      "min_eligible_ma_per_op": 1,  # the minimum number of machines capable to process an operation
      "max_eligible_ma_per_op": 6,  # the maximum number of machines capable to process an operation
    },
)

In [8]:
# from rl4co.envs import ENV_REGISTRY

# ENV_REGISTRY["fjspt"] = FJSPTEnv

# env = FJSPTEnv(
#     generator_params={
#      "num_jobs": 3,  # the total number of jobs
#       "num_machines": 2,  # the total number of machines that can process operations
#       "num_trucks": 1,  # the total number of trucks
#       "min_ops_per_job": 2,  # minimum number of operatios per job
#       "max_ops_per_job": 3,  # maximum number of operations per job
#       "min_processing_time": 1,  # the minimum time required for a machine to process an operation
#       "max_processing_time": 5,  # the maximum time required for a machine to process an operation
#       "min_transportation_time": 0,  # the minimum time required for a truck to transport
#       "max_transportation_time": 5,  # the maximum time required for a truck to transport
#       "min_eligible_ma_per_op": 1,  # the minimum number of machines capable to process an operation
#       "max_eligible_ma_per_op": 2,  # the maximum number of machines capable to process an operation
#     },
# )

In [9]:
accelerator = "gpu"
batch_size = 1024
max_epochs = 50
train_data_size = batch_size * 100
val_data_size = 1_000
test_data_size = 1_000
embed_dim = 128
num_encoder_layers = 4

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Политика диспетчеризации (Learning to Dispatch):
# нейросеть encoder–decoder, выбирающая следующее допустимое назначение операции на машину и транспортное средство
policy = L2DPolicy(embed_dim=embed_dim, num_encoder_layers=num_encoder_layers)
policy = policy.to(device)

# RL-модель Learning to Dispatch:
# обучает политику методом REINFORCE с rollout-бейзлайном,
# оптимизируя makespan через взаимодействие с FJSP-симуляцией
model = L2DModel(env,
                 policy=policy, 
                 baseline="rollout",
                 batch_size=batch_size,
                 train_data_size=train_data_size,
                 val_data_size=val_data_size,
                 test_data_size=test_data_size,
                 optimizer_kwargs={"lr": 1e-4})

/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'env' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['env'])`.
/home/egor/cousework/rl4co_try/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'policy' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['policy'])`.


## Тестируем greedy rollout baseline c необученной моделью

**Интерактивное построение решения**

In [ ]:
policy.eval()
td = env.reset(batch_size=[1]).to(device)
td1 = td.clone()

with torch.no_grad():
    out = policy(td1, env, phase="test", decode_type="greedy", return_actions=True)

td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([385], device='cuda:0') tensor([45.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([364], device='cuda:0') tensor([39.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([349], device='cuda:0') tensor([120.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([245], device='cuda:0') tensor([88.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([365], device='cuda:0') tensor([150.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([386], device='cuda:0') tensor([144.], device='cuda:0')
td["machine_finish_times"][batch_idx, machine_op] = tensor([0]) tensor([291], device='cuda:0') tensor([174.], device='cuda:0')


In [ ]:
fig = None
env.render(td1.cpu(), 0)
plt.show()

actions = out["actions"]
print(actions)

for step in range(actions.shape[1]):
    td1["action"] = actions[:, step]
    td1 = env.step(td1)["next"]

    clear_output(wait=True)
    if fig is not None:
        plt.close(fig)
    fig = plt.figure()
    _, machine_schedule, trucks_schedule, ops_debug, machine_conflicts, truck_conflicts, job_conflicts = env.render(td1.cpu(), 0)
    plt.show()
    time.sleep(1)

In [9]:
from validation import validate_solution
validate_solution(td1, idx=0, verbose=True)

VALID SOLUTION


True

In [10]:
machine_schedule

defaultdict(list,
            {0: [(1, 384.0, 415.0),
              (21, 305.0, 326.0),
              (22, 0.0, 37.0),
              (23, 79.0, 102.0),
              (25, 195.0, 234.0)],
             1: [(4, 529.0, 561.0),
              (14, 298.0, 315.0),
              (16, 407.0, 435.0),
              (17, 52.0, 69.0),
              (24, 136.0, 163.0)],
             2: [(0, 355.0, 370.0), (5, 165.0, 204.0), (13, 225.0, 249.0)],
             3: [(10, 158.0, 176.0), (11, 199.0, 238.0), (15, 351.0, 376.0)],
             4: [(6, 278.0, 310.0), (7, 323.0, 344.0), (18, 108.0, 142.0)],
             5: [(9, 65.0, 85.0), (19, 185.0, 215.0), (20, 240.0, 263.0)],
             6: [(3, 487.0, 502.0),
              (8, 379.0, 404.0),
              (26, 268.0, 299.0),
              (28, 80.0, 103.0)],
             7: [(2, 444.0, 473.0), (12, 130.0, 161.0), (27, 14.0, 51.0)]})

In [11]:
trucks_schedule

defaultdict(list,
            {1: [(6, 0.0, 14.0),
              (2, 37.0, 65.0),
              (5, 65.0, 79.0),
              (4, 79.0, 108.0),
              (5, 108.0, 136.0),
              (1, 136.0, 165.0),
              (5, 165.0, 195.0),
              (1, 249.0, 278.0),
              (4, 278.0, 305.0),
              (1, 310.0, 323.0),
              (3, 323.0, 351.0),
              (1, 351.0, 379.0),
              (3, 379.0, 407.0),
              (0, 415.0, 444.0),
              (0, 473.0, 487.0)],
             0: [(5, 0.0, 0.0),
              (4, 37.0, 52.0),
              (6, 52.0, 80.0),
              (3, 102.0, 130.0),
              (2, 130.0, 158.0),
              (4, 158.0, 185.0),
              (2, 185.0, 199.0),
              (3, 199.0, 225.0),
              (4, 225.0, 240.0),
              (5, 240.0, 268.0),
              (3, 268.0, 298.0),
              (0, 326.0, 355.0),
              (0, 370.0, 384.0),
              (0, 502.0, 529.0)]})

In [12]:
last_op_finish_times = td1["machine_finish_times"].gather(1, td1["end_op_per_job"])
print("Makespan:", last_op_finish_times.max().item())

Makespan: 561.0


In [13]:
actions_untrained = out['actions'].cpu().detach()
rewards_untrained = out['reward'].cpu().detach()
print(f"Cost: {-rewards_untrained[0]}")

Cost: 561.0


## Обучение

In [7]:
policy.train()

L2DPolicy(
  (encoder): HetGNNEncoder(
    (init_embedding): FJSPTInitEmbedding(
      (init_ops_embed): Linear(in_features=5, out_features=128, bias=False)
      (pos_encoder): PositionalEncoding(
        (dropout): Dropout(p=0.0, inplace=False)
      )
      (init_ma_embed): Linear(in_features=1, out_features=128, bias=False)
      (init_truck_embed): Linear(in_features=1, out_features=128, bias=False)
      (proc_edge_embed): Linear(in_features=1, out_features=128, bias=False)
      (truck_edge_embed): Linear(in_features=1, out_features=128, bias=False)
    )
    (layers): ModuleList(
      (0-3): 4 x HetGNNBlock(
        (ma_from_ops): HetGNNLayer(
          (activation): ReLU()
        )
        (ops_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ma_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (truck_from_ma): HetGNNLayer(
          (activation): ReLU()
        )
        (ffn_ma): TransformerFFN(
          (ops): ModuleDict(
   

In [8]:
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints",
    filename="reinforce_epoch_{epoch:03d}",
    save_top_k=1,
    save_last=True,
    monitor="val/reward",
    mode="max",  # maximize reward = minimize makespan
)

rich_model_summary = RichModelSummary(max_depth=3)

callbacks = [checkpoint_callback, rich_model_summary]

In [9]:
from comet_key import API_KEY

logger = CometLogger(
    api_key=API_KEY,
    project="rl4co_reinforce",
)

trainer = RL4COTrainer(
    max_epochs=max_epochs,
    accelerator=accelerator,
    devices=1,
    logger=logger,
    callbacks=callbacks,
)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/fsoidhfoi/rl4co-reinforce/24c0203a5cef45719727c43a2b0af5b3

Using 16bit Automatic Mixed Precision (AMP)
Trainer already configured with model summary callbacks: [<class 'lightning.pytorch.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [10]:
trainer.fit(model)

val_file not set. Generating dataset instead
test_file not set. Generating dataset instead


AssertionError: 

## Тестирование обученной модели

In [ ]:
policy.eval()
policy.to(device) # добавил и ошибка исчезла ???? почему ?????
td2 = td.clone()

with torch.no_grad():
    out = policy(td2.clone(), phase="test", decode_type="greedy", return_actions=True)

In [ ]:
fig = None
env.render(td2.cpu(), 0)
plt.show()

actions = out["actions"]

for step in range(actions.shape[1]):
    td2["action"] = actions[:, step]
    td2 = env.step(td2)["next"]

    clear_output(wait=True)
    if fig is not None:
        plt.close(fig)
    fig = plt.figure()
    env.render(td2.cpu(), 0)
    plt.show()
    time.sleep(1)

In [ ]:
last_op_finish_times = td2["machine_finish_times"].gather(1, td2["end_op_per_job"])
print("Makespan:", last_op_finish_times.max().item())

In [ ]:
actions_untrained = out['actions'].cpu().detach()
rewards_untrained = out['reward'].cpu().detach()
print(f"Cost: {-rewards_untrained[0]}")

In [ ]:
trainer.test(model)